# LeetCode 2806 — Account Balance After Rounded Purchase bằng Python

Link đề: https://leetcode.com/problems/account-balance-after-rounded-purchase/ · Độ khó: **Easy** · Chủ đề: Math, Làm tròn số

Skeleton LeetCode:

```python
class Solution:
    def accountBalanceAfterPurchase(self, purchaseAmount: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: tài khoản ban đầu có **100 đô**, mua hàng giá `purchaseAmount` nhưng thực tế bị trừ theo số tiền đã **làm tròn tới bội số của 10 gần nhất** (hoà thì chọn bội số **lớn hơn**).
2. Biết **tự tính tay** việc làm tròn: nhìn chữ số hàng đơn vị để quyết định làm tròn xuống hay lên.
3. Học các công cụ Python: `//` (chia nguyên), `%` (chia dư), `abs`, `min` với `key`, `range` có bước nhảy, và vì sao **không nên** dùng `round()` ở bài này.
4. Đi từ **Brute Force (thử mọi bội số của 10)** → **`min` với khoá phụ** → **dùng chia dư `% 10`** → **công thức một dòng `(x + 5) // 10 * 10`**.
5. Chứng minh công thức đúng, kiểm tra bằng vét cạn toàn miền `0..100`, và tránh các lỗi kinh điển: làm tròn hoà sai phía, dùng `round()` (làm tròn kiểu ngân hàng), quên trừ khỏi 100.

## 2. Đề bài & ví dụ

Ban đầu tài khoản ngân hàng của bạn có **100 đô**.

Bạn được cho một số nguyên `purchaseAmount` — số tiền bạn định chi cho một món hàng. Khi thanh toán, cửa hàng sẽ làm tròn `purchaseAmount` thành `roundedAmount`, là **bội số của 10 gần nhất** với `purchaseAmount`. Nếu có **nhiều hơn một** bội số của 10 cùng gần nhất (tức là bị "hoà"), thì chọn **bội số lớn hơn**.

Trả về một số nguyên: **số dư còn lại** trong tài khoản sau giao dịch, tức `100 - roundedAmount`.

Lưu ý: `0` cũng được xem là một bội số của 10.

| Ví dụ | `purchaseAmount` | Output | Giải thích |
|---|---|---|---|
| 1 | `9` | `90` | Bội số của 10 gần 9 nhất là 10 (cách 1), nên `roundedAmount = 10`, số dư `100 − 10 = 90`. |
| 2 | `15` | `80` | 15 cách 10 là 5, cách 20 là 5: hoà, chọn bội số lớn hơn là 20. Số dư `100 − 20 = 80`. |

Constraints:

- `0 <= purchaseAmount <= 100`

In [1]:
# Chạy thử "bằng tay" hai ví dụ của đề
for p in (9, 15):
    # các bội số của 10 gần nhất ở hai phía
    thap = p // 10 * 10
    cao = thap + 10
    print(f"p={p}: bội số dưới = {thap} (cách {p - thap}), bội số trên = {cao} (cách {cao - p})")

p=9: bội số dưới = 0 (cách 9), bội số trên = 10 (cách 1)
p=15: bội số dưới = 10 (cách 5), bội số trên = 20 (cách 5)


## 3. Kiến thức nền

Bài này nhỏ nhưng dùng vài công cụ Python rất hay gặp. Ta học từng cái với ví dụ nhỏ.

### 3.1. Phép chia nguyên `//` và phép chia dư `%`

- `a // b` là **thương nguyên** (bỏ phần thập phân, làm tròn xuống với số dương): `37 // 10 = 3`.
- `a % b` là **số dư**: `37 % 10 = 7`. Với `b = 10`, đây chính là **chữ số hàng đơn vị**.
- Luôn có `a == (a // b) * b + a % b`.

Vậy `37 // 10 * 10 = 30` là **bội số của 10 ngay dưới 37** (làm tròn xuống), còn `37 % 10 = 7` cho biết ta đã "đi quá" bội số đó bao xa.

In [2]:
print(37 // 10, 37 % 10)        # 3 7
print(37 // 10 * 10)            # 30  -> bội số của 10 ngay dưới 37
print(40 // 10 * 10, 40 % 10)   # 40 0  -> đã là bội số rồi thì giữ nguyên
a, b = 37, 10
print(a == (a // b) * b + a % b)  # True

3 7
30
40 0
True


### 3.2. `range(start, stop, step)` — dãy có bước nhảy

`range(0, 101, 10)` sinh ra `0, 10, 20, ..., 100` (dừng **trước** `stop = 101`). Đây chính là danh sách mọi bội số của 10 trong đoạn `[0, 100]`, rất tiện để thử từng ứng viên.

In [3]:
print(list(range(0, 101, 10)))
print(len(range(0, 101, 10)))   # 11 bội số

[0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
11


### 3.3. `abs(x)` — giá trị tuyệt đối (khoảng cách)

`abs(x)` bỏ dấu của `x`. Khoảng cách giữa hai số `p` và `m` trên trục số là `abs(p - m)`, không phụ thuộc số nào lớn hơn.

In [4]:
print(abs(9 - 10), abs(10 - 9))   # 1 1
print(abs(15 - 10), abs(15 - 20)) # 5 5  -> bị "hoà"

1 1
5 5


### 3.4. `min(iterable, key=...)` và so sánh tuple

`min(danh_sách, key=f)` trả về phần tử có `f(phần_tử)` **nhỏ nhất**. Nếu `f` trả về **tuple**, Python so sánh **từng phần tử từ trái sang phải**: so phần tử đầu, nếu bằng nhau mới so phần tử sau. Đây là cách rất gọn để diễn đạt "ưu tiên 1 là gần nhất, ưu tiên 2 (khi hoà) là lớn hơn".

In [5]:
print((5, -20) < (5, -10))        # True: bằng nhau ở phần tử đầu, -20 < -10
ung_vien = [10, 20]
p = 15
best = min(ung_vien, key=lambda m: (abs(m - p), -m))
print(best)                        # 20: hoà khoảng cách thì lấy m lớn hơn (vì -m nhỏ hơn)

True
20


### 3.5. Phép `round()` của Python — cẩn thận!

`round(2.5)` trả về `2` (không phải 3), vì Python dùng **làm tròn kiểu ngân hàng** (round-half-to-even): khi hoà thì chọn số **chẵn**. Điều này **sai** với yêu cầu "hoà thì chọn lớn hơn". Ta sẽ không dùng `round()` cho bài này.

In [6]:
print(round(0.5), round(1.5), round(2.5), round(3.5))   # 0 2 2 4
print(round(15 / 10), round(25 / 10))                   # 2 2  (25/10 = 2.5 -> 2, sai so với đề)

0 2 2 4
2 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Hãy tưởng tượng một **trục số** có các vạch to ở mỗi bội số của 10: `..., 0, 10, 20, 30, ...`. Số `purchaseAmount` nằm giữa hai vạch liên tiếp: vạch dưới `lo` và vạch trên `hi = lo + 10`.

| `purchaseAmount` | `lo` | `hi` | cách `lo` | cách `hi` | chọn |
|---|---|---|---|---|---|
| 9 | 0 | 10 | 9 | 1 | 10 |
| 15 | 10 | 20 | 5 | 5 | 20 (hoà ⇒ lớn hơn) |
| 14 | 10 | 20 | 4 | 6 | 10 |
| 20 | 20 | 30 | 0 | 10 | 20 |
| 100 | 100 | 110 | 0 | 10 | 100 |

**Nhận xét:** nếu gọi `r = purchaseAmount % 10` (chữ số hàng đơn vị) thì cách `lo` là `r` và cách `hi` là `10 − r`.

- `r < 5` ⇒ gần `lo` hơn ⇒ làm tròn **xuống**.
- `r > 5` ⇒ gần `hi` hơn ⇒ làm tròn **lên**.
- `r == 5` ⇒ hoà ⇒ theo đề chọn **lên**.

Gộp lại: **`r >= 5` thì làm tròn lên, còn lại làm tròn xuống.**

In [7]:
for p in (0, 4, 5, 9, 14, 15, 20, 99, 100):
    r = p % 10
    huong = "lên" if r >= 5 else "xuống"
    print(f"p={p:3}: chữ số đơn vị r={r} -> làm tròn {huong}")

p=  0: chữ số đơn vị r=0 -> làm tròn xuống
p=  4: chữ số đơn vị r=4 -> làm tròn xuống
p=  5: chữ số đơn vị r=5 -> làm tròn lên
p=  9: chữ số đơn vị r=9 -> làm tròn lên
p= 14: chữ số đơn vị r=4 -> làm tròn xuống
p= 15: chữ số đơn vị r=5 -> làm tròn lên
p= 20: chữ số đơn vị r=0 -> làm tròn xuống
p= 99: chữ số đơn vị r=9 -> làm tròn lên
p=100: chữ số đơn vị r=0 -> làm tròn xuống


## 5. Cách 1 — Brute Force / baseline: thử mọi bội số của 10

**Ý tưởng:** liệt kê mọi bội số của 10 trong khoảng có thể (từ `0` đến `100`, ta cho tới `110` cho an toàn), với mỗi ứng viên tính khoảng cách tới `purchaseAmount`, giữ ứng viên **gần nhất**. Khi gặp ứng viên có khoảng cách **bằng** khoảng cách tốt nhất hiện tại thì ta cũng chấp nhận (vì duyệt tăng dần, ứng viên sau là bội số **lớn hơn**).

Hai điểm tinh tế:

1. Duyệt theo thứ tự **tăng dần**.
2. So sánh bằng `<=` (không phải `<`) để khi hoà, ứng viên lớn hơn (xuất hiện sau) **ghi đè**.

In [8]:
from typing import List

def balance_bruteforce(purchaseAmount: int) -> int:
    best = None            # bội số gần nhất tìm được
    best_dist = None       # khoảng cách của nó tới purchaseAmount
    for m in range(0, 111, 10):          # 0, 10, ..., 110
        d = abs(m - purchaseAmount)
        if best is None or d <= best_dist:   # <= : hoà thì lấy m lớn hơn
            best = m
            best_dist = d
    return 100 - best

In [9]:
print(balance_bruteforce(9))    # 90
print(balance_bruteforce(15))   # 80
print(balance_bruteforce(0))    # 100
print(balance_bruteforce(100))  # 0

90
80
100
0


**Độ phức tạp:** số ứng viên là hằng số (12 bội số) ⇒ Time `O(1)`; nếu tổng quát hoá cho `purchaseAmount` tới `X`, ta duyệt `X/10` ứng viên ⇒ `O(X)`. Space `O(1)`.

**Nút thắt:** ta đang duyệt cả những ứng viên rõ ràng quá xa. Thực ra chỉ **hai** ứng viên có thể thắng: bội số ngay dưới và ngay trên. Biết điều đó ta không cần vòng lặp.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — chỉ có 2 ứng viên.** Với `lo = purchaseAmount // 10 * 10` và `hi = lo + 10`, mọi bội số khác của 10 đều xa `purchaseAmount` hơn ít nhất một trong hai số này.

**Đột phá 2 — dùng chữ số hàng đơn vị.** `r = purchaseAmount % 10`. Nếu `r >= 5` chọn `hi`, ngược lại chọn `lo`. Điều kiện `>=` (không phải `>`) chính là luật "hoà ⇒ lớn hơn".

**Đột phá 3 — công thức một dòng.** Cộng thêm `5` trước khi chia nguyên:

`roundedAmount = (purchaseAmount + 5) // 10 * 10`

Vì sao? `purchaseAmount + 5` đẩy những số có `r >= 5` qua vạch tiếp theo: ví dụ `15 + 5 = 20 → 20 // 10 * 10 = 20`; còn `14 + 5 = 19 → 19 // 10 * 10 = 10`. Đây là mẹo làm tròn nửa-lên kinh điển.

In [10]:
for p in (9, 14, 15, 20, 25, 99, 100):
    c = (p + 5) // 10 * 10
    print(f"p={p:3}: (p+5)={p + 5:3} -> (p+5)//10*10 = {c:3} -> số dư = {100 - c}")

p=  9: (p+5)= 14 -> (p+5)//10*10 =  10 -> số dư = 90
p= 14: (p+5)= 19 -> (p+5)//10*10 =  10 -> số dư = 90
p= 15: (p+5)= 20 -> (p+5)//10*10 =  20 -> số dư = 80
p= 20: (p+5)= 25 -> (p+5)//10*10 =  20 -> số dư = 80
p= 25: (p+5)= 30 -> (p+5)//10*10 =  30 -> số dư = 70
p= 99: (p+5)=104 -> (p+5)//10*10 = 100 -> số dư = 0
p=100: (p+5)=105 -> (p+5)//10*10 = 100 -> số dư = 0


## 7. Các cách tiếp theo

Ta giảng thêm 3 cách nữa, đi từ "ngắn gọn bằng `min`" tới "công thức".

### Cách 2 — `min` với khoá `(khoảng cách, −giá trị)`

**Từng đoạn:**

- `range(0, 101, 10)`: tất cả bội số của 10 trong `[0, 100]`.
- `key=lambda m: (abs(m - p), -m)`: ưu tiên gần nhất; nếu hoà, `-m` nhỏ hơn tương ứng `m` **lớn hơn**.
- `min(...)` trả về bội số tốt nhất ⇒ số dư là `100 - best`.

In [11]:
def balance_min_key(purchaseAmount: int) -> int:
    best = min(range(0, 101, 10), key=lambda m: (abs(m - purchaseAmount), -m))
    return 100 - best

In [12]:
print(balance_min_key(9))    # 90
print(balance_min_key(15))   # 80
print(balance_min_key(100))  # 0

90
80
0


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def accountBalanceAfterPurchase(self, purchaseAmount: int) -> int:
        best = min(range(0, 101, 10), key=lambda m: (abs(m - purchaseAmount), -m))
        return 100 - best
```

**Độ phức tạp:** Time `O(1)` (11 ứng viên cố định; `O(X/10)` nếu mở rộng miền), Space `O(1)` (`min` duyệt lười trên `range`, không tạo danh sách).

### Cách 3 — Dùng chia dư `% 10` (2 ứng viên)

**Từng đoạn:**

- `r = purchaseAmount % 10`: phần lẻ so với bội số dưới.
- `lo = purchaseAmount - r`: bội số của 10 ngay dưới (hoặc bằng chính nó).
- `r >= 5` ⇒ `lo + 10`, ngược lại `lo`.

In [13]:
def balance_mod(purchaseAmount: int) -> int:
    r = purchaseAmount % 10
    lo = purchaseAmount - r
    rounded = lo + 10 if r >= 5 else lo
    return 100 - rounded

In [14]:
print(balance_mod(9))    # 90
print(balance_mod(15))   # 80
print(balance_mod(14))   # 90

90
80
90


**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def accountBalanceAfterPurchase(self, purchaseAmount: int) -> int:
        r = purchaseAmount % 10
        rounded = purchaseAmount - r + (10 if r >= 5 else 0)
        return 100 - rounded
```

**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 4 — Công thức `(x + 5) // 10 * 10`

**Từng đoạn:** cộng 5 để "nửa trên" đẩy qua vạch kế tiếp; `// 10` lấy số vạch; `* 10` đổi lại thành số tiền. Đây là cách ngắn nhất và là **code nộp** của ta.

In [15]:
def balance_formula(purchaseAmount: int) -> int:
    return 100 - (purchaseAmount + 5) // 10 * 10

In [16]:
print(balance_formula(9))    # 90
print(balance_formula(15))   # 80
print(balance_formula(5))    # 90 (5 hoà giữa 0 và 10 -> chọn 10)

90
80
90


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

Lưu ý: ngoài `purchaseAmount` là số nguyên không âm (đúng constraints), công thức dùng `//` vẫn đúng vì không có số âm.

## 8. TRACE từng bước

Ta trace hai thuật toán: Cách 1 (duyệt ứng viên) trên `purchaseAmount = 15` để thấy khi nào "hoà" và ai thắng, và Cách 4 (công thức) từng bước số học.

Bảng trace Cách 1 với `p = 15`:

| ứng viên `m` | `abs(m − 15)` | `best_dist` trước | có `d <= best_dist`? | `best` sau |
|---|---|---|---|---|
| 0 | 15 | — | (khởi tạo) | 0 |
| 10 | 5 | 15 | có (5 ≤ 15) | 10 |
| 20 | 5 | 5 | **có (5 ≤ 5, hoà)** | **20** |
| 30 | 15 | 5 | không | 20 |
| 40 | 25 | 5 | không | 20 |
| ... | tăng | 5 | không | 20 |

In [17]:
def trace_bruteforce(purchaseAmount):
    best, best_dist = None, None
    for m in range(0, 61, 10):
        d = abs(m - purchaseAmount)
        if best is None or d <= best_dist:
            hanh_dong = "cập nhật" + (" (hoà -> lấy lớn hơn)" if d == best_dist else "")
            best, best_dist = m, d
        else:
            hanh_dong = "bỏ qua"
        print(f"m={m:3}, d={d:3} -> {hanh_dong:22} | best={best}, best_dist={best_dist}")
    print("rounded =", best, "-> số dư =", 100 - best)
    return 100 - best

trace_bruteforce(15)

m=  0, d= 15 -> cập nhật               | best=0, best_dist=15
m= 10, d=  5 -> cập nhật               | best=10, best_dist=5
m= 20, d=  5 -> cập nhật (hoà -> lấy lớn hơn) | best=20, best_dist=5
m= 30, d= 15 -> bỏ qua                 | best=20, best_dist=5
m= 40, d= 25 -> bỏ qua                 | best=20, best_dist=5
m= 50, d= 35 -> bỏ qua                 | best=20, best_dist=5
m= 60, d= 45 -> bỏ qua                 | best=20, best_dist=5
rounded = 20 -> số dư = 80


80

In [18]:
def trace_formula(purchaseAmount):
    plus5 = purchaseAmount + 5
    vach = plus5 // 10
    rounded = vach * 10
    print(f"p + 5 = {purchaseAmount} + 5 = {plus5}")
    print(f"(p + 5) // 10 = {vach}   (đây là 'số vạch' của bội số 10)")
    print(f"rounded = {vach} * 10 = {rounded}")
    print("số dư =", 100 - rounded)
    return 100 - rounded

trace_formula(15)
print("---")
trace_formula(14)
print("---")
trace_formula(100)

p + 5 = 15 + 5 = 20
(p + 5) // 10 = 2   (đây là 'số vạch' của bội số 10)
rounded = 2 * 10 = 20
số dư = 80
---
p + 5 = 14 + 5 = 19
(p + 5) // 10 = 1   (đây là 'số vạch' của bội số 10)
rounded = 1 * 10 = 10
số dư = 90
---
p + 5 = 100 + 5 = 105
(p + 5) // 10 = 10   (đây là 'số vạch' của bội số 10)
rounded = 10 * 10 = 100
số dư = 0


0

## 9. Code nộp LeetCode

Dùng Cách 4 (công thức): ngắn nhất, `O(1)`, dễ chứng minh.

In [19]:
from typing import List

class Solution:
    def accountBalanceAfterPurchase(self, purchaseAmount: int) -> int:
        roundedAmount = (purchaseAmount + 5) // 10 * 10
        return 100 - roundedAmount

In [20]:
print(Solution().accountBalanceAfterPurchase(9))    # 90
print(Solution().accountBalanceAfterPurchase(15))   # 80

90
80


## 10. Vì sao thuật toán đúng

**Bước 1 — chỉ hai bội số có thể thắng.** Gọi `lo = 10k` là bội số lớn nhất `<= p` và `hi = 10(k + 1)`. Mọi bội số `m <= lo − 10` có `p − m >= p − lo + 10 > p − lo`; mọi `m >= hi + 10` có `m − p >= hi + 10 − p > hi − p`. Vậy chỉ cần so `lo` và `hi`.

**Bước 2 — so sánh `lo` và `hi`.** Đặt `r = p − lo` với `0 <= r <= 9`. Khoảng cách tới `lo` là `r`, tới `hi` là `10 − r`.

- `r < 5` ⇒ `r < 10 − r` ⇒ chọn `lo`.
- `r > 5` ⇒ `r > 10 − r` ⇒ chọn `hi`.
- `r = 5` ⇒ hoà ⇒ đề yêu cầu chọn bội số lớn hơn, tức `hi`.

Tóm lại chọn `hi` **khi và chỉ khi** `r >= 5`.

**Bước 3 — công thức `(p + 5) // 10 * 10` khớp.** Viết `p = 10k + r`. Khi đó `p + 5 = 10k + (r + 5)`:

- `r <= 4` ⇒ `r + 5 <= 9` ⇒ `(p + 5) // 10 = k` ⇒ kết quả `10k = lo`. Khớp.
- `r >= 5` ⇒ `10 <= r + 5 <= 14` ⇒ `(p + 5) // 10 = k + 1` ⇒ kết quả `10(k + 1) = hi`. Khớp.

**Bước 4 — số dư.** Đề yêu cầu `100 − roundedAmount`; vì `0 <= p <= 100` nên `roundedAmount <= 100` và đáp án không âm.

## 11. Phân tích Time / Space Complexity

Gọi `X` là giá trị lớn nhất của `purchaseAmount` (ở đây `X = 100`, hằng số).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute Force thử ứng viên | `O(X)` (hằng số 12 lần) | `O(1)` | dễ nghĩ, mô phỏng đúng đề |
| 2. `min` với key | `O(X)` (hằng số 11 lần) | `O(1)` | ngắn gọn nhưng vẫn duyệt |
| 3. Chia dư `% 10` | `O(1)` | `O(1)` | 2 ứng viên |
| 4. Công thức | `O(1)` | `O(1)` | ngắn nhất |

Với miền `0..100`, mọi cách đều chạy tức thì. Điểm khác nhau chỉ là độ gọn và khả năng mở rộng khi miền lớn (số đến `10^9` thì chỉ cách 3, 4 dùng được).

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: `0` (không mua gì, bản thân là bội số của 10); `4` (làm tròn xuống về 0); `5` (hoà giữa 0 và 10, chọn 10); `9`; `10`, `20` (đã là bội số); `14`/`15` (biên của hoà); `95`, `99`; `100` (tối đa, số dư `0`).

In [21]:
impls = {
    "bruteforce": balance_bruteforce,
    "min_key": balance_min_key,
    "mod": balance_mod,
    "formula": balance_formula,
    "Solution": Solution().accountBalanceAfterPurchase,
}
cases = [
    ("ví dụ 1", 9, 90),
    ("ví dụ 2", 15, 80),
    ("p = 0", 0, 100),
    ("p = 4 xuống 0", 4, 100),
    ("p = 5 hoà chọn 10", 5, 90),
    ("p = 10 đã là bội số", 10, 90),
    ("p = 14", 14, 90),
    ("p = 25 hoà chọn 30", 25, 70),
    ("p = 95 hoà chọn 100", 95, 0),
    ("p = 99", 99, 0),
    ("p = 100", 100, 0),
]
for name, p, exp in cases:
    for k, f in impls.items():
        got = f(p)
        assert got == exp, (name, k, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 90
PASS ví dụ 2 -> 80
PASS p = 0 -> 100
PASS p = 4 xuống 0 -> 100
PASS p = 5 hoà chọn 10 -> 90
PASS p = 10 đã là bội số -> 90
PASS p = 14 -> 90
PASS p = 25 hoà chọn 30 -> 70
PASS p = 95 hoà chọn 100 -> 0
PASS p = 99 -> 0
PASS p = 100 -> 0


### 12.2. Vét cạn toàn miền `0..100`

Miền hợp lệ chỉ có 101 giá trị nên ta kiểm **tất cả**. Dùng làm "oracle" một hàm tự viết trực tiếp từ định nghĩa bằng phân số (so sánh khoảng cách tới `lo` và `hi`).

In [22]:
def oracle(p):
    lo = p // 10 * 10
    hi = lo + 10
    if p - lo < hi - p:
        rounded = lo
    else:          # gần hi hơn hoặc hoà -> hi
        rounded = hi
    return 100 - rounded

for p in range(0, 101):
    exp = oracle(p)
    for k, f in impls.items():
        assert f(p) == exp, (p, k)
print("101 giá trị: mọi cách khớp oracle")

101 giá trị: mọi cách khớp oracle


### 12.3. Kiểm tính chất và ngẫu nhiên có seed

Tính chất cần đúng: đáp án luôn trong `[0, 100]`, `100 − đáp án` là bội số của 10, độ lệch `|p − rounded| <= 5`, và đáp án **không tăng** khi `purchaseAmount` tăng.

In [23]:
import random
random.seed(2806)
for _ in range(300):
    p = random.randint(0, 100)
    ans = Solution().accountBalanceAfterPurchase(p)
    rounded = 100 - ans
    assert 0 <= ans <= 100
    assert rounded % 10 == 0
    assert abs(p - rounded) <= 5
    if p < 100:
        assert Solution().accountBalanceAfterPurchase(p + 1) <= ans
    for f in impls.values():
        assert f(p) == ans
print("300 ca ngẫu nhiên: các tính chất đều đúng")

300 ca ngẫu nhiên: các tính chất đều đúng


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Dùng `round()` của Python

In [24]:
def sai_round(purchaseAmount: int) -> int:
    return 100 - round(purchaseAmount / 10) * 10    # sai: làm tròn ngân hàng

print("sai :", sai_round(25))                           # round(2.5) = 2 -> 80
print("đúng:", Solution().accountBalanceAfterPurchase(25))  # 70
print("sai :", sai_round(5), "| đúng:", Solution().accountBalanceAfterPurchase(5))   # 100 vs 90

sai : 80
đúng: 70
sai : 100 | đúng: 90


**Vì sao sai:** `round(2.5)` trả `2` vì Python làm tròn hoà về số chẵn. Đề yêu cầu hoà thì lấy bội số **lớn hơn** (`25` phải về `30`). Hãy tự cài đặt quy tắc bằng `% 10` hoặc `+ 5`.

### Lỗi 2: Hoà làm tròn xuống (dùng `> 5` thay `>= 5`)

In [25]:
def sai_hoa_xuong(purchaseAmount: int) -> int:
    r = purchaseAmount % 10
    rounded = purchaseAmount - r + (10 if r > 5 else 0)     # sai: thiếu dấu =
    return 100 - rounded

print("sai :", sai_hoa_xuong(15))                           # 90
print("đúng:", Solution().accountBalanceAfterPurchase(15))  # 80

sai : 90
đúng: 80


**Vì sao sai:** khi `r = 5`, khoảng cách tới hai bội số bằng nhau nên đề bắt chọn bội số lớn hơn. Phải dùng `r >= 5`.

### Lỗi 3: Chỉ làm tròn xuống (dùng `//` đơn thuần)

In [26]:
def sai_chi_xuong(purchaseAmount: int) -> int:
    return 100 - purchaseAmount // 10 * 10     # sai: luôn làm tròn xuống

print("sai :", sai_chi_xuong(9))                            # 100
print("đúng:", Solution().accountBalanceAfterPurchase(9))   # 90

sai : 100
đúng: 90


**Vì sao sai:** `//` luôn làm tròn **xuống**, nhưng `9` gần `10` hơn `0`. Cần cộng bù `5` trước khi chia.

### Lỗi 4: Quên trừ khỏi 100 (trả về `roundedAmount`)

In [27]:
def sai_quen_tru(purchaseAmount: int) -> int:
    return (purchaseAmount + 5) // 10 * 10      # sai: trả về số tiền trừ

print("sai :", sai_quen_tru(15))                            # 20
print("đúng:", Solution().accountBalanceAfterPurchase(15))  # 80

sai : 20
đúng: 80


**Vì sao sai:** đề hỏi **số dư còn lại** `100 − roundedAmount`, không phải số tiền bị trừ.

### Lỗi 5 (nhỏ): Nghĩ `0` không phải bội số của 10

`0` là bội số của 10 (`0 = 10 × 0`). Với `purchaseAmount = 0..4`, kết quả làm tròn là `0` và số dư `100`. Nếu code của bạn loại ứng viên `0` thì sẽ sai ở các ca này.

In [28]:
print([Solution().accountBalanceAfterPurchase(p) for p in range(0, 6)])   # [100, 100, 100, 100, 100, 90]

[100, 100, 100, 100, 100, 90]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề "làm tròn tới bội số của `k` gần nhất, hoà thì lên" ⇒ công thức **`(x + k // 2) // k * k`** (với `k` chẵn). Ở đây `k = 10` nên cộng 5.
- Đề "làm tròn **xuống** bội số của `k`" ⇒ `x // k * k`; "làm tròn **lên**" ⇒ `(x + k − 1) // k * k`. Nhớ ba mẫu này.
- Khi thấy chữ "hoà" trong đề, **luôn** kiểm biên hoà bằng các ví dụ như `5`, `15`, `25`.
- Đừng dùng `round()` của Python cho bài làm tròn có quy tắc riêng: nó làm tròn kiểu ngân hàng.
- Miền nhỏ (`0..100`) ⇒ có thể vét cạn để kiểm công thức, rất an toàn.

## 15. Biến thể của bài

1. **Số dư ban đầu khác:** thay `100` bằng `balance` bất kỳ.
2. **Làm tròn tới bội số của `k`:** `roundedAmount = (x + k // 2) // k * k` (hoà lên). Thử `k = 5, 100`.
3. **Hoà làm tròn xuống:** dùng `(x + (k − 1) // 2) // k * k` hoặc điều kiện `r > k / 2`.
4. **Nhiều giao dịch liên tiếp:** cho mảng `purchases`, trừ lần lượt, dừng khi số dư không đủ.
5. **Làm tròn "về phía số chẵn của vạch"** (round half to even) đúng chuẩn ngân hàng: so sánh với `round()` ở trên.
6. **Số âm:** nếu `x` âm thì `//` làm tròn xuống về `−∞`; cần cẩn thận về định nghĩa "hoà".

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute Force | thử mọi bội số, giữ gần nhất (hoà lấy lớn) | `O(X)` | `O(1)` | làm oracle, người mới |
| 2. `min` + key | khoá `(khoảng cách, −m)` | `O(X)` | `O(1)` | khi muốn code ngắn mà rõ nghĩa |
| 3. Chia dư | 2 ứng viên `lo`, `hi` dựa vào `r >= 5` | `O(1)` | `O(1)` | dễ giải thích với người mới |
| 4. Công thức | `(x + 5) // 10 * 10` | `O(1)` | `O(1)` | **code nộp**, nhanh, ngắn |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `47 // 10`, `47 % 10`, `(47 + 5) // 10 * 10`, `min([10, 20], key=lambda m: (abs(m - 15), -m))`.

**Bài 2 (tính tay).** Tính số dư tài khoản (ban đầu 100) khi `purchaseAmount` lần lượt là `0, 4, 5, 36, 65, 99`.

**Bài 3 (điền TODO).** Hoàn thành `balance_todo(p)` theo Cách 3 (chia dư).

In [29]:
def balance_todo(p):
    # TODO 1: r = chữ số hàng đơn vị của p
    # TODO 2: lo = bội số của 10 ngay dưới (hoặc bằng) p
    # TODO 3: nếu r >= 5 thì rounded = lo + 10, ngược lại rounded = lo
    # TODO 4: return 100 - rounded
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `balance_bruteforce2(p)` bằng vòng `for m in range(0, 101, 10)` và so sánh khoảng cách, **không nhìn** code mẫu. Chú ý điều kiện khi hoà.

**Bài 5 (biến thể / mini challenge).** (a) Viết `round_to_multiple(x, k)` làm tròn `x` tới bội số của `k` gần nhất, hoà thì lên. (b) Viết `total_after_purchases(purchases)` bắt đầu từ `100`, trừ liên tiếp các số tiền đã làm tròn tới bội số của 10, nếu giao dịch nào vượt quá số dư thì bỏ qua giao dịch đó.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `47 // 10 = 4`, `47 % 10 = 7`, `52 // 10 = 5`. Bài 2: nhìn chữ số hàng đơn vị `r`; `r >= 5` thì lên.
- **Mức 2 (vừa):** Bài 3: bốn dòng đã ghi trong comment TODO. Bài 4: duyệt `m` tăng dần, giữ `best`; khi `d <= best_dist` thì cập nhật.
- **Mức 3 (sâu):** Bài 5(a): `(x + k // 2) // k * k` đúng khi `k` chẵn; với `k` lẻ không có "hoà" nên dùng `(x + k // 2) // k * k` vẫn đúng. Bài 5(b): dùng vòng `for` và `if` so sánh `rounded <= balance` trước khi trừ.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert 47 // 10 == 4 and 47 % 10 == 7
assert (47 + 5) // 10 * 10 == 50
assert min([10, 20], key=lambda m: (abs(m - 15), -m)) == 20

# Bài 2: số dư
exp2 = {0: 100, 4: 100, 5: 90, 36: 60, 65: 30, 99: 0}
for p, e in exp2.items():
    assert Solution().accountBalanceAfterPurchase(p) == e, p

# Bài 3
def balance_todo_ans(p):
    r = p % 10
    lo = p - r
    rounded = lo + 10 if r >= 5 else lo
    return 100 - rounded

for p in range(0, 101):
    assert balance_todo_ans(p) == Solution().accountBalanceAfterPurchase(p)

# Bài 4
def balance_bruteforce2(p):
    best, best_dist = 0, abs(0 - p)
    for m in range(0, 101, 10):
        d = abs(m - p)
        if d <= best_dist:
            best, best_dist = m, d
    return 100 - best

for p in range(0, 101):
    assert balance_bruteforce2(p) == Solution().accountBalanceAfterPurchase(p)

# Bài 5(a)
def round_to_multiple(x, k):
    return (x + k // 2) // k * k

assert round_to_multiple(15, 10) == 20
assert round_to_multiple(14, 10) == 10
assert round_to_multiple(7, 5) == 5
assert round_to_multiple(8, 5) == 10
assert round_to_multiple(149, 100) == 100
assert round_to_multiple(150, 100) == 200

# Bài 5(b)
def total_after_purchases(purchases):
    balance = 100
    for p in purchases:
        rounded = (p + 5) // 10 * 10
        if rounded <= balance:
            balance -= rounded
    return balance

assert total_after_purchases([9, 15]) == 70      # 100 -10 -20
assert total_after_purchases([55, 55]) == 40     # 100 -60 ; 60 > 40 nên bỏ
assert total_after_purchases([]) == 100
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (làm tròn tới bội số của 10 gần nhất, hoà lên, rồi trừ khỏi 100)

```python
from typing import List

class Solution:
    def accountBalanceAfterPurchase(self, purchaseAmount: int) -> int:
        roundedAmount = (purchaseAmount + 5) // 10 * 10   # làm tròn nửa-lên
        return 100 - roundedAmount                         # số dư
```

**Checklist**

- [ ] Hiểu rằng `0` cũng là bội số của 10?
- [ ] Hoà (chữ số đơn vị `5`) thì làm tròn **lên**?
- [ ] Không dùng `round()` của Python (làm tròn ngân hàng)?
- [ ] Trả về **số dư** `100 − roundedAmount`, không phải số tiền bị trừ?
- [ ] Đã thử `p = 0, 4, 5, 9, 15, 95, 100`?
- [ ] Đã so công thức với vét cạn toàn miền `0..100`?
- [ ] Signature đúng `accountBalanceAfterPurchase(self, purchaseAmount: int) -> int`?